# 02 — Learn mode connections

Load previously learned teacher trajectories, optimise a quadratic Bézier control point between the initial and final checkpoint of each expert, and save the resulting mode connections.

The dataset selector uses the same unambiguous names as the teacher notebook, so MIMIC-IHM and MIMIC-phenotyping can no longer overwrite one another.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "btm").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from btm.loaders import DATASET_CHOICES, load_dataset
from btm.models import default_net_type, get_model
from btm.utils import *


## Configuration


In [ ]:
DATASET = "eicu"          # "eicu", "mimic3_ihm", "mimic3_ph"
NET_TYPE = None
OPTIM = "sgd"
TRAIN_BATCH = 256
SEED = 42
EXPERT_TO_INSPECT = 0

NET_TYPE = NET_TYPE or default_net_type(DATASET)
set_seed(SEED)
device = get_device()


## Load data and teacher trajectories


In [ ]:
bundle = load_dataset(
    DATASET,
    data_root=ROOT / "data",
    train_batch=TRAIN_BATCH,
    pre_process="std",
)
train_loader = bundle.train_loader
val_loader = bundle.val_loader

paths = experiment_paths(DATASET, NET_TYPE, OPTIM, root=ROOT)
trajectories = torch.load(paths["trajectory"], map_location="cpu", weights_only=False)
print(f"loaded {len(trajectories)} expert trajectories from {paths['trajectory']}")

model_factory = lambda: get_model(
    DATASET, NET_TYPE, device=device, output_dim=bundle.output_dim
)
loss_fn = nn.BCEWithLogitsLoss()
mode_cfg = get_mode_hyperparameters(DATASET)
mode_cfg


## Inspect one expert and fit one Bézier curve


In [ ]:
record = trajectories[EXPERT_TO_INSPECT]
trajectory = record["trajectory"]
theta_A, theta_B = trajectory[0], trajectory[-1]
model = model_factory()

sgd_train_losses = []
sgd_val_losses = []
for checkpoint in trajectory:
    model.load_state_dict(checkpoint)
    sgd_train_losses.append(
        evaluate_loss(model, train_loader, loss_fn, device, task=bundle.task)
    )
    sgd_val_losses.append(
        evaluate_loss(model, val_loader, loss_fn, device, task=bundle.task)
    )

single_result = learn_low_loss_bezier_curve(
    model_factory(),
    train_loader,
    theta_A,
    theta_B,
    loss_fn,
    device=device,
    **mode_cfg,
)


In [ ]:
theta_C = single_result["theta_C"]
num_points = len(trajectory)

t_values, train_bezier = evaluate_curve_grid(
    model_factory(), train_loader, loss_fn,
    theta_A, theta_B, theta_C,
    num_points=num_points, device=device, curve_type="bezier",
)
_, train_linear = evaluate_curve_grid(
    model_factory(), train_loader, loss_fn,
    theta_A, theta_B, None,
    num_points=num_points, device=device, curve_type="linear",
)
_, val_bezier = evaluate_curve_grid(
    model_factory(), val_loader, loss_fn,
    theta_A, theta_B, theta_C,
    num_points=num_points, device=device, curve_type="bezier",
)
_, val_linear = evaluate_curve_grid(
    model_factory(), val_loader, loss_fn,
    theta_A, theta_B, None,
    num_points=num_points, device=device, curve_type="linear",
)

x_interp = (len(trajectory) - 1) * t_values
plt.figure(figsize=(9, 4))
plt.plot(range(len(sgd_val_losses)), sgd_val_losses, label="teacher trajectory")
plt.plot(x_interp, val_bezier, label="Bézier")
plt.plot(x_interp, val_linear, label="linear")
plt.xlabel("epoch-equivalent position")
plt.ylabel("validation loss")
plt.legend()
plt.grid(True)
plt.show()


## Learn all mode connections


In [ ]:
modes = learn_mode_connections(
    trajectories=trajectories,
    model_factory=model_factory,
    train_loader=train_loader,
    loss_fn=loss_fn,
    device=device,
    **mode_cfg,
)


## Save


In [ ]:
paths["trajectory_dir"].mkdir(parents=True, exist_ok=True)
torch.save(modes, paths["modes"])
print("saved:", paths["modes"])
